# LangChain document loaders: classroom lesson
Load TXT, PDF, CSV, web and Wikipedia. Local loading needs no API key. Run cells from top to bottom.

**Colab:** upload the supplied classroom ZIP in the next cell. **VS Code:** open the extracted folder and select your `.venv` notebook kernel; skip the Colab upload cell.

In [ ]:
# Colab only: upload the classroom ZIP supplied with this lesson.
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
zip_names = [name for name in uploaded if name.endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError('Upload exactly one classroom ZIP.')
with zipfile.ZipFile(zip_names[0]) as archive:
    # Only extract the supplied classroom archive.
    target = Path('/content/classroom').resolve()
    for member in archive.namelist():
        if not (target / member).resolve().is_relative_to(target):
            raise ValueError('Unsafe ZIP path.')
    archive.extractall(target)
os.chdir(target / 'langchain_loader_classroom')
print('Project folder:', Path.cwd())

In [ ]:
# Install the lesson dependencies. In VS Code, select the .venv kernel first.
%pip install -q -r requirements.txt

## 1. Imports and paths
The shared Python file contains commented functions used by this notebook. In VS Code, start with the project folder open. Restart the kernel after installation if it asks you to.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os
assert Path('data').is_dir(), 'Open the project folder or run the Colab upload cell.'
load_dotenv('.env')
from loaders_demo import show_documents, load_online_examples, summarize_documents
from langchain_community.document_loaders import TextLoader, PyPDFLoader, CSVLoader
from itertools import islice
DATA_DIR = Path('data').resolve()
print(DATA_DIR)

## 2. TextLoader
One text file becomes one Document. Inspect its text and metadata.

In [ ]:
text_docs = TextLoader(str(DATA_DIR / "successful_dairy_farming.txt"), encoding="utf-8").load()
show_documents("Dairy farming text", text_docs)

## 3. PyPDFLoader
Page mode produces five Documents. A PDF table becomes extracted text; this does not reconstruct a spreadsheet.

In [ ]:
pdf_docs = PyPDFLoader(str(DATA_DIR / "pakistan_dairy_meat_report.pdf"), mode="page").load()
show_documents("PDF pages", pdf_docs)
assert len(pdf_docs) == 5

## 4. CSVLoader
Each row becomes a Document. The supplied CSV has 30,000 rows; start with five.

In [ ]:
csv_loader = CSVLoader(str(DATA_DIR / "employee_records.csv"), encoding="utf-8-sig")
csv_docs = list(islice(csv_loader.lazy_load(), 5))
show_documents("First five CSV rows", csv_docs)
# All rows, if needed: all_csv_docs = csv_loader.load()

## 5. Optional web and Wikipedia loaders
Run only when internet is available. No AI key is needed. These examples can fail if a website changes or blocks requests.

In [ ]:
# Uncomment to run the online examples.
# load_online_examples()

## 6. Optional Gemini or Groq summary
Loading and AI generation are separate steps. Choose one provider. In Colab, add `GOOGLE_API_KEY` or `GROQ_API_KEY` under the key icon (Secrets), and grant this notebook access. Locally, copy `.env.example` to `.env` and fill one key. Only the dairy TXT is sent in this example; the employee CSV is not sent. Provider charges and quotas may apply.

In [ ]:
PROVIDER = 'gemini'  # Change to 'groq' to use Groq.
RUN_AI = False       # Change to True when you want to make an API request.
if RUN_AI:
    key_name = 'GOOGLE_API_KEY' if PROVIDER == 'gemini' else 'GROQ_API_KEY'
    if not os.getenv(key_name):
        try:
            from google.colab import userdata
            os.environ[key_name] = userdata.get(key_name)
        except ImportError:
            from getpass import getpass
            os.environ[key_name] = getpass(f'Enter {key_name}: ')
        except Exception:
            raise ValueError(f'Add {key_name} to Colab Secrets and grant notebook access.') from None
    # Set GEMINI_MODEL or GROQ_MODEL in os.environ to change the model.
    try:
        print(summarize_documents(text_docs, PROVIDER))
    except Exception as error:
        print('Summary failed:', type(error).__name__)
        print('Check key, model availability, internet, billing and quota.')
else:
    print('AI step skipped. All local loader examples work without a key.')

## Teaching exercises
1. Print the second PDF page and its metadata.
2. Preview ten CSV rows instead of five.
3. Explain the difference between `load()` and `lazy_load()`.
4. Compare the same dairy summary across providers if both are available.
5. Explain why higher national production does not prove higher profit per farm.